### 1. Configuração do Ambiente

In [16]:
#!pip install "cognite-sdk[pandas]" matplotlib seaborn tensorflow plotly -q

import os
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display
from sklearn.preprocessing import RobustScaler
from statsmodels.tsa.stattools import acf
from industrial_ts.dataloader import DataLoader
import json
from getpass import getpass
#import tensorflow as tf
#from tensorflow import keras

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [5]:
import debugpy
debugpy.listen(('0.0.0.0', 5678))

('0.0.0.0', 5678)

In [17]:
os.environ['COGNITE_CLIENT_SECRET'] = getpass("Enter COGNITE_CLIENT")

### 2. Ativar o DataLoader

In [18]:
import importlib
import sys
importlib.reload(sys.modules['industrial_ts.dataloader'])
from industrial_ts.dataloader import DataLoader
dl = DataLoader()
dl.add_segments(segments=3, window=10, step=10, series=[
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActShaft Power',
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActPress Ratio'  
], path="segmenter_model_10.pkl"
)
#dl.segmenter.save("segmenter_model_10.pkl")



Buscando dados para as 12 séries temporais encontradas.


### 10 Pós-processamento

In [29]:
for i in [2]:
    dl.df['states'].replace(i, 1, inplace=True) # Merge states 1 and 2


/tmp/ipykernel_43074/3961075041.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dl.df['states'].replace(i, 1, inplace=True) # Merge states 1 and 2


In [30]:
dl.add_time_to_change_state_timestamp()

### Geração seqKAN

In [32]:
from industrial_ts.seqKAN import TSDF_seqKAN

In [35]:
kan_params = {
    'hidden': {'grid': 5, 'k': 3, 'grid_range': (-4, 4)},
    'output': {'grid': 5, 'k': 3, 'grid_range': (-4, 4)},
}

model = TSDF_seqKAN(
    in_channels=11,
    hidden_dim=11*32,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
        ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    #status_dim=3,
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params,
)

/home/ferna/fe/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [37]:
res = model.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=256,
    window_size=10,
    window_step=10,
    epochs = 500,
    validate=False,
    patience=50,
    kl_warmup_epochs=20,
    kl_start=0.01,
    rebuild=False,
    reconstruction_test=False,
    warmup_steps=0,
    min_lr_factor=1,
    optimizer_name='adam',
    optimizer_params={"lr": 5e-5}
)
res = [r for r in res if r is not None]  # remove None results
with open("final_rebuild_gru_.json", "w") as f:
    json.dump(res, f)

/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/industrial_ts/tsdiffusion.py:693: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ds.tensors = (torch.tensor(x_all_scaled, dtype=torch.float32),) + ds.tensors[1:]


GRUPOS (total): {0: 24346, 1: 2637}
GRUPOS (train): {0: 19477, 1: 2110}
GRUPOS (test):  {0: 4869, 1: 527}
Epoch 1/500 | Train(sampled) L1:1.466012 L2:0.000000 L3:0.000000 L4:0.000000 L5:0.000000 L6:0.000000 | 
          >> Test macro:0.617819 ± 0.172166 | micro:0.479929 ± 0.174658
Epoch 2/500 | Train(sampled) L1:1.068558 L2:0.000000 L3:0.000000 L4:0.000000 L5:0.000000 L6:0.000000 | 
          >> Test macro:0.628060 ± 0.190508 | micro:0.475481 ± 0.173987
Epoch 3/500 | Train(sampled) L1:1.364084 L2:0.000000 L3:0.000000 L4:0.000000 L5:0.000000 L6:0.000000 | 
          >> Test macro:0.620788 ± 0.162691 | micro:0.490487 ± 0.174022
Epoch 4/500 | Train(sampled) L1:1.462682 L2:0.000000 L3:0.000000 L4:0.000000 L5:0.000000 L6:0.000000 | 
          >> Test macro:0.610373 ± 0.165460 | micro:0.477855 ± 0.173922
Epoch 5/500 | Train(sampled) L1:1.455174 L2:0.000000 L3:0.000000 L4:0.000000 L5:0.000000 L6:0.000000 | 
          >> Test macro:0.614614 ± 0.165434 | micro:0.482117 ± 0.173954
Epoch 6/500 | 

KeyboardInterrupt: 